# JEDI Ridge -> XGBoost Stacked K-Fold Workbook

**Objective:** predict personal-fit scores with Ridge, append those predicted scores, then train XGBoost for `applied`.

**Input:** `data/jedi_master_prepared.csv`

**Main evaluation:** leakage-safe 5-fold CV.

## Step 0: Imports And Paths

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    mean_absolute_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "xgboost is not installed in this kernel. Run '%pip install xgboost', restart the kernel, then rerun."
    ) from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
N_SPLITS = 5


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "jedi_master_prepared.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/jedi_master_prepared.csv from this notebook location.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "jedi_master_prepared.csv"
OUTPUT_DIR = PROJECT_ROOT / "data"

print(PROJECT_ROOT)
print(DATA_PATH)

## Step 1: Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head(3))
display(df["applied"].value_counts().rename_axis("applied").reset_index(name="rows"))

## Step 2: Define Targets

In [ ]:
SCORE_TARGETS = [
    "role_title_preference_score",
    "role_family_preference_score",
    "seniority_profile_fit_score",
    "location_profile_fit_score",
    "work_mode_fit_score",
    "experience_profile_fit_score",
    "skills_profile_fit_score",
    "salary_profile_fit_score",
    "overall_preference_score",
]

missing_score_targets = [c for c in SCORE_TARGETS if c not in df.columns]
if missing_score_targets:
    raise ValueError(f"Missing score target columns: {missing_score_targets}")

df["applied_clean"] = df["applied"].astype(str).str.strip().str.lower()
bad_applied = sorted(set(df["applied_clean"]) - {"yes", "no"})
if bad_applied:
    raise ValueError(f"Applied column has invalid values: {bad_applied}")

df["target_applied"] = df["applied_clean"].map({"no": 0, "yes": 1}).astype(int)

for col in SCORE_TARGETS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid_rows = df["target_applied"].notna() & df[SCORE_TARGETS].notna().all(axis=1)
model_df = df.loc[valid_rows].reset_index(drop=True).copy()

y_applied = model_df["target_applied"].astype(int)
y_scores = model_df[SCORE_TARGETS].astype(float)

score_audit = pd.DataFrame({
    "target": SCORE_TARGETS,
    "min": [y_scores[c].min() for c in SCORE_TARGETS],
    "max": [y_scores[c].max() for c in SCORE_TARGETS],
    "mean": [y_scores[c].mean() for c in SCORE_TARGETS],
    "missing": [model_df[c].isna().sum() for c in SCORE_TARGETS],
})

display(score_audit)
display(y_applied.value_counts().rename_axis("target_applied").reset_index(name="rows"))

## Step 3: Build Leakage-Free Raw Features

In [ ]:
ALL_SCORE_COLUMNS = [
    c for c in model_df.columns
    if c.endswith("_score") or c in ["company_match_score", "overall_preference_score"]
]

LEAKAGE_OR_METADATA_COLUMNS = [
    "row_id",
    "applied",
    "applied_clean",
    "target_applied",
    "preference_reason",
    "apply_url",
    "source_platform",
    "source_type",
    "scraped_at",
    "near_duplicate_key",
    "company_name",
    "company_name_standard",
    "role_title_standard",
]

drop_columns = sorted(set(ALL_SCORE_COLUMNS + LEAKAGE_OR_METADATA_COLUMNS))
available_drop_columns = [c for c in drop_columns if c in model_df.columns]

meta_columns = [
    "row_id",
    "company_name",
    "role_title",
    "role_family",
    "seniority",
    "location",
    "applied",
    "source_type",
]
meta = model_df[[c for c in meta_columns if c in model_df.columns]].copy()

candidate_raw_features = [c for c in model_df.columns if c not in available_drop_columns]

display(pd.DataFrame({"dropped_from_training_X": available_drop_columns}))
display(pd.DataFrame({"candidate_raw_feature": candidate_raw_features}))

## Step 4: Feature Groups

In [ ]:
text_source_columns = [
    "role_title",
    "role_family",
    "seniority",
    "location",
    "skills_or_requirements",
    "skills_or_requirements_original",
    "skills_array",
]

categorical_features = [
    "can_apply_now",
    "freshness",
    "role_family_standard",
    "seniority_standard",
    "employment_type_standard",
    "work_mode_standard",
    "city_standard",
    "country_standard",
    "location_bucket",
    "experience_min_bucket",
    "skills_imputation_type",
]

numeric_features = [
    "experience_min_years",
    "salary_min",
    "salary_max",
    "salary_midpoint",
]

boolean_features = [
    "has_skills_or_requirements",
    "is_bengaluru",
    "is_gurgaon_ncr",
    "is_india",
    "salary_available",
]

text_source_columns = [c for c in text_source_columns if c in model_df.columns]
categorical_features = [c for c in categorical_features if c in model_df.columns]
numeric_features = [c for c in numeric_features if c in model_df.columns]
boolean_features = [c for c in boolean_features if c in model_df.columns]

for col in numeric_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")


def to_binary(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return 1.0
    if text in {"false", "0", "no"}:
        return 0.0
    return np.nan


for col in boolean_features:
    model_df[col] = model_df[col].map(to_binary)

model_df["job_text"] = (
    model_df[text_source_columns]
    .fillna("")
    .astype(str)
    .agg(" | ".join, axis=1)
)

X_raw = model_df[candidate_raw_features].copy()
X_raw["job_text"] = model_df["job_text"]

display(pd.DataFrame({
    "feature_group": ["text_source", "categorical", "numeric", "boolean"],
    "columns": [
        ", ".join(text_source_columns),
        ", ".join(categorical_features),
        ", ".join(numeric_features),
        ", ".join(boolean_features),
    ],
}))

## Step 5: Pipelines

In [ ]:
def make_preprocess(extra_numeric_features=None, text_max_features=1800):
    extra_numeric_features = list(extra_numeric_features or [])
    numeric_cols = numeric_features + boolean_features + extra_numeric_features

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
    ])

    text_transformer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=text_max_features,
        sublinear_tf=True,
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_cols),
            ("categorical", categorical_transformer, categorical_features),
            ("text", text_transformer, "job_text"),
        ],
        remainder="drop",
    )


def make_score_model(alpha=2.0):
    return Pipeline(steps=[
        ("preprocess", make_preprocess(text_max_features=2200)),
        ("regressor", Ridge(alpha=alpha)),
    ])


def make_xgb_model(extra_numeric_features=None, scale_pos_weight=1.0):
    return Pipeline(steps=[
        ("preprocess", make_preprocess(extra_numeric_features=extra_numeric_features, text_max_features=1800)),
        ("classifier", XGBClassifier(
            n_estimators=350,
            max_depth=3,
            learning_rate=0.035,
            min_child_weight=4,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.10,
            reg_lambda=3.00,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=4,
            scale_pos_weight=scale_pos_weight,
        )),
    ])


SCORE_BOUNDS = {
    "salary_profile_fit_score": (0.0, 8.0),
}


def clip_scores(predictions):
    predictions = np.asarray(predictions, dtype=float).copy()
    for idx, target in enumerate(SCORE_TARGETS):
        low, high = SCORE_BOUNDS.get(target, (0.0, 5.0))
        predictions[:, idx] = np.clip(predictions[:, idx], low, high)
    return predictions


PRED_SCORE_COLUMNS = [f"pred_{c}" for c in SCORE_TARGETS]
TRUE_SCORE_COLUMNS = [f"true_{c}" for c in SCORE_TARGETS]


def append_score_columns(X_base, score_array, prefix):
    X_aug = X_base.copy()
    score_array = clip_scores(score_array)
    for idx, score_name in enumerate(SCORE_TARGETS):
        X_aug[f"{prefix}_{score_name}"] = score_array[:, idx]
    return X_aug

## Step 6: Metrics

In [ ]:
def regression_metrics(y_true, y_pred, split_name, fold=None):
    y_true_df = pd.DataFrame(np.asarray(y_true, dtype=float), columns=SCORE_TARGETS)
    y_pred_df = pd.DataFrame(clip_scores(y_pred), columns=SCORE_TARGETS)
    rows = []
    for target in SCORE_TARGETS:
        actual = y_true_df[target]
        pred = y_pred_df[target]
        abs_error = (pred - actual).abs()
        rows.append({
            "fold": fold,
            "split": split_name,
            "target": target,
            "mae": mean_absolute_error(actual, pred),
            "rmse": np.sqrt(np.mean(np.square(pred - actual))),
            "r2": r2_score(actual, pred),
            "spearman": actual.corr(pred, method="spearman"),
            "within_0_5": (abs_error <= 0.5).mean(),
            "within_1_0": (abs_error <= 1.0).mean(),
        })
    return pd.DataFrame(rows)


def classification_summary(model_name, fold, y_true, proba, threshold=0.50):
    proba = np.asarray(proba, dtype=float)
    pred = (proba >= threshold).astype(int)
    return {
        "model": model_name,
        "fold": fold,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_yes": precision_score(y_true, pred, zero_division=0),
        "recall_yes": recall_score(y_true, pred, zero_division=0),
        "f1_yes": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "average_precision": average_precision_score(y_true, proba),
        "log_loss": log_loss(y_true, proba, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, proba),
        "predicted_yes_count": int(pred.sum()),
    }


def threshold_table(y_true, proba, model_name):
    rows = []
    for threshold in np.arange(0.20, 0.81, 0.05):
        pred = (np.asarray(proba) >= threshold).astype(int)
        rows.append({
            "model": model_name,
            "threshold": round(float(threshold), 2),
            "predicted_yes_count": int(pred.sum()),
            "precision_yes": precision_score(y_true, pred, zero_division=0),
            "recall_yes": recall_score(y_true, pred, zero_division=0),
            "f1_yes": f1_score(y_true, pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        })
    return pd.DataFrame(rows)

## Step 7: Leakage-Safe K-Fold Run

In [ ]:
def run_stacked_xgb_cv():
    outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    classification_rows = []
    regression_rows = []
    prediction_frames = []

    for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(X_raw, y_applied), start=1):
        X_tr_raw = X_raw.iloc[train_idx].copy()
        X_val_raw = X_raw.iloc[val_idx].copy()
        y_tr = y_applied.iloc[train_idx].copy()
        y_val = y_applied.iloc[val_idx].copy()
        y_tr_scores = y_scores.iloc[train_idx].copy()
        y_val_scores = y_scores.iloc[val_idx].copy()

        scale_pos_weight = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))

        fold_score_model = make_score_model(alpha=2.0)
        train_score_oof = cross_val_predict(
            fold_score_model,
            X_tr_raw,
            y_tr_scores,
            cv=inner_cv,
            n_jobs=None,
        )
        train_score_oof = clip_scores(train_score_oof)

        fold_score_model.fit(X_tr_raw, y_tr_scores)
        val_score_pred = clip_scores(fold_score_model.predict(X_val_raw))

        regression_rows.append(regression_metrics(y_tr_scores, train_score_oof, "train_inner_oof", fold=fold_idx))
        regression_rows.append(regression_metrics(y_val_scores, val_score_pred, "outer_validation", fold=fold_idx))

        X_tr_stacked = append_score_columns(X_tr_raw, train_score_oof, prefix="pred")
        X_val_stacked = append_score_columns(X_val_raw, val_score_pred, prefix="pred")
        X_tr_oracle = append_score_columns(X_tr_raw, y_tr_scores.values, prefix="true")
        X_val_oracle = append_score_columns(X_val_raw, y_val_scores.values, prefix="true")

        fold_configs = [
            ("raw_xgboost", make_xgb_model(scale_pos_weight=scale_pos_weight), X_tr_raw, X_val_raw),
            ("stacked_ridge_scores_xgboost", make_xgb_model(PRED_SCORE_COLUMNS, scale_pos_weight=scale_pos_weight), X_tr_stacked, X_val_stacked),
            ("oracle_true_scores_xgboost", make_xgb_model(TRUE_SCORE_COLUMNS, scale_pos_weight=scale_pos_weight), X_tr_oracle, X_val_oracle),
        ]

        fold_predictions = meta.iloc[val_idx].reset_index(drop=True).copy()
        fold_predictions["fold"] = fold_idx
        fold_predictions["actual_applied"] = y_val.reset_index(drop=True).values

        for target_idx, target in enumerate(SCORE_TARGETS):
            fold_predictions[f"true_{target}"] = y_val_scores.reset_index(drop=True)[target]
            fold_predictions[f"pred_{target}"] = val_score_pred[:, target_idx]

        for model_name, estimator, X_tr, X_val in fold_configs:
            estimator.fit(X_tr, y_tr)
            proba = estimator.predict_proba(X_val)[:, 1]
            classification_rows.append(classification_summary(model_name, fold_idx, y_val, proba, threshold=0.50))
            fold_predictions[f"{model_name}_probability"] = proba
            fold_predictions[f"{model_name}_prediction"] = (proba >= 0.50).astype(int)

        prediction_frames.append(fold_predictions)

    return (
        pd.DataFrame(classification_rows),
        pd.concat(regression_rows, ignore_index=True),
        pd.concat(prediction_frames, ignore_index=True).sort_values(["fold", "row_id"]).reset_index(drop=True),
    )


cv_detail, score_cv_detail, oof_predictions = run_stacked_xgb_cv()

display(cv_detail)
display(score_cv_detail.head(20))
display(oof_predictions.head(10))

## Step 8: Aggregate CV Results

In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_yes",
    "recall_yes",
    "f1_yes",
    "roc_auc",
    "average_precision",
    "log_loss",
    "brier_score",
    "predicted_yes_count",
]

cv_summary = (
    cv_detail
    .groupby("model")[metric_columns]
    .agg(["mean", "std"])
)

flat_cv_summary = cv_summary.copy()
flat_cv_summary.columns = [f"{metric}_{stat}" for metric, stat in flat_cv_summary.columns]
flat_cv_summary = flat_cv_summary.reset_index().sort_values("f1_yes_mean", ascending=False)

display(cv_summary)
display(flat_cv_summary)

score_cv_summary = (
    score_cv_detail
    .groupby(["split", "target"])[["mae", "rmse", "r2", "spearman", "within_0_5", "within_1_0"]]
    .agg(["mean", "std"])
)
display(score_cv_summary)

## Step 9: Threshold Review

In [ ]:
threshold_frames = []
for model_name in [
    "raw_xgboost",
    "stacked_ridge_scores_xgboost",
    "oracle_true_scores_xgboost",
]:
    threshold_frames.append(threshold_table(
        oof_predictions["actual_applied"],
        oof_predictions[f"{model_name}_probability"],
        model_name,
    ))

threshold_review = pd.concat(threshold_frames, ignore_index=True)
display(threshold_review.sort_values(["model", "f1_yes"], ascending=[True, False]).groupby("model").head(5))

## Step 10: Save Artifacts

In [ ]:
cv_detail_path = OUTPUT_DIR / "jedi_xgboost_stacked_outer_cv_detail.csv"
cv_summary_path = OUTPUT_DIR / "jedi_xgboost_stacked_outer_cv_summary.csv"
score_cv_detail_path = OUTPUT_DIR / "jedi_xgboost_stacked_ridge_score_cv_detail.csv"
oof_predictions_path = OUTPUT_DIR / "jedi_xgboost_stacked_oof_predictions.csv"
threshold_review_path = OUTPUT_DIR / "jedi_xgboost_stacked_threshold_review.csv"

cv_detail.to_csv(cv_detail_path, index=False)
flat_cv_summary.to_csv(cv_summary_path, index=False)
score_cv_detail.to_csv(score_cv_detail_path, index=False)
oof_predictions.to_csv(oof_predictions_path, index=False)
threshold_review.to_csv(threshold_review_path, index=False)

saved = pd.DataFrame({
    "artifact": [
        "cv_detail",
        "cv_summary",
        "ridge_score_cv_detail",
        "oof_predictions",
        "threshold_review",
    ],
    "path": [
        str(cv_detail_path),
        str(cv_summary_path),
        str(score_cv_detail_path),
        str(oof_predictions_path),
        str(threshold_review_path),
    ],
})

display(saved)

## Step 11: Readout

In [ ]:
best_model = flat_cv_summary.iloc[0]["model"]
best_f1 = flat_cv_summary.iloc[0]["f1_yes_mean"]
best_auc = flat_cv_summary.iloc[0]["roc_auc_mean"]

print(f"Best CV model by F1_yes: {best_model}")
print(f"F1_yes mean: {best_f1:.4f}")
print(f"ROC-AUC mean: {best_auc:.4f}")

display(
    oof_predictions
    .sort_values("stacked_ridge_scores_xgboost_probability", ascending=False)
    .head(25)
    [[
        "row_id",
        "company_name",
        "role_title",
        "actual_applied",
        "stacked_ridge_scores_xgboost_probability",
        "raw_xgboost_probability",
        "oracle_true_scores_xgboost_probability",
    ] + [f"pred_{c}" for c in SCORE_TARGETS]]
)